# Overview

Middleware cung cấp một phương pháp để kiểm soát chặt chẽ hơn những gì diễn ra bên trong agent. Middleware rất hữu ích cho các trường hợp sau:

* Theo dõi hành vi của agent thông qua logging, analytic và debugging.
* Biến đổi các prompt, [lựa chọn tool](https://docs.langchain.com/oss/python/langchain/middleware/built-in#llm-tool-selector), và định dạng dữ liệu đầu ra.
* Thêm [retry](https://docs.langchain.com/oss/python/langchain/middleware/built-in#tool-retry), [fallback](https://docs.langchain.com/oss/python/langchain/middleware/built-in#model-fallback), và logic kết thúc sớm.
* Áp dụng [rate limit](https://docs.langchain.com/oss/python/langchain/middleware/built-in#model-call-limit), guardrail, và [phát hiện PII](https://docs.langchain.com/oss/python/langchain/middleware/built-in#pii-detection) (thông tin định danh cá nhân).

Thêm middleware bằng cách truyền chúng vào [`create_agent`](https://reference.langchain.com/python/langchain/agents/factory/create_agent):

```python
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware

agent = create_agent(
    model="gpt-5.5",
    tools=[...],
    middleware=[
        SummarizationMiddleware(...),
        HumanInTheLoopMiddleware(...)
    ],
)
```

## Vòng lặp agent

Vòng lặp cốt lõi của agent bao gồm việc gọi một model, cho phép nó chọn các tool để thực thi, và sau đó kết thúc khi nó không gọi thêm tool nào nữa:

<p align="center">
    <img src="https://mintcdn.com/langchain-5e9cc07a/Tazq8zGc0yYUYrDl/oss/images/core_agent_loop.png?fit=max&auto=format&n=Tazq8zGc0yYUYrDl&q=85&s=ac72e48317a9ced68fd1be64e89ec063" height="300">
</p>

Middleware cung cấp các hook trước và sau mỗi bước nói trên:

<p align="center">
    <img src="https://mintcdn.com/langchain-5e9cc07a/RAP6mjwE5G00xYsA/oss/images/middleware_final.png?fit=max&auto=format&n=RAP6mjwE5G00xYsA&q=85&s=eb4404b137edec6f6f0c8ccb8323eaf1" height="400">
</p>

## Sử dụng middleware bên trong workflow của LangGraph

Middleware không phải là một môi trường thực thi độc lập: các hook chạy bên trong [LangGraph](/oss/python/langgraph/overview) đã được biên dịch mà [`create_agent`](https://reference.langchain.com/python/langchain/agents/factory/create_agent) trả về. Bạn có thể đưa toàn bộ agent (bao gồm cả middleware và mọi thứ đi kèm) vào một [StateGraph](https://reference.langchain.com/python/langgraph/graph/state/StateGraph) lớn hơn dưới dạng một node hoặc subgraph, và mọi hook của middleware vẫn sẽ tiếp tục hoạt động.

Hãy áp dụng pattern này khi cấu trúc tổng thể phức tạp hơn một vòng lặp "chạy đến khi hoàn tất" thông thường: ví dụ như phân loại đầu vào trước khi điều hướng đến một trong số nhiều agent, phân tán công việc để xử lý song song, hoặc liên kết các lệnh gọi agent lại với nhau bằng các bước tất định.

`HumanInTheLoopMiddleware` thực hiện đối chiếu dựa trên thuộc tính `.name` của từng tool.

Các hàm sử dụng decorator `@tool` sẽ tự động lấy tên từ chính tên hàm đó, vì vậy key trong ví dụ dưới đây là `"send_email"`.

```python
from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.graph import START, StateGraph

# Giả định rằng read_email, send_email, classify_node và route đã được định nghĩa ở nơi khác.
email_agent = create_agent(
    model="claude-sonnet-4-6",
    tools=[read_email, send_email],
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"send_email": True})],
)

graph = (
    StateGraph(AgentState)
    .add_node("classify", classify_node)
    .add_node("email_agent", email_agent)
    .add_edge(START, "classify")
    .add_conditional_edges("classify", route)
    .compile()
)
```

Các tính năng như ngắt HITL (Human-in-the-Loop), tóm tắt văn bản, che giấu thông tin PII, retry, và bất kỳ hook tùy chỉnh nào cũng đều gắn liền với agent node. Xem thêm [Sử dụng subgraph](/oss/python/langgraph/use-subgraphs) để nắm được toàn bộ các mẫu cấu trúc, bao gồm cả phạm vi hoạt động của checkpointer trong subgraph (theo từng lần gọi hay theo từng luồng).

## Tài nguyên bổ sung

* **Middleware tích hợp sẵn**: Khám phá các middleware được tích hợp sẵn cho các use case phổ biến.
* **Middleware tùy chỉnh**: Tự xây dựng middleware của riêng bạn bằng các hook và decorator.
* **Tài liệu API Middleware**: Tài liệu tham khảo API đầy đủ về middleware.
* **Tích hợp middleware**: Các middleware dành riêng cho các nhà cung cấp như Anthropic, AWS, OpenAI, và nhiều nền tảng khác.
* **Kiểm thử agent**: Kiểm thử các agent của bạn bằng LangSmith.